# KPI's de RRHH

## Importación de librerías

In [73]:
import pandas as pd
import numpy as np

## Carga de los datos

In [74]:
df = pd.read_csv("/kaggle/input/datasets/leonardo117/employees-data/employees.csv")
print(df.head(5))

   employee_id gender  birth_date   hire_date termination_date department  \
0            1      F  1967-10-29  2015-11-15              NaN         IT   
1            2      M  1970-11-09  2018-09-30       2024-07-13      Sales   
2            3      M  1998-01-20  2014-01-12              NaN    Finance   
3            4      M  1993-06-02  2023-05-10              NaN         HR   
4            5      M  1975-01-31  2011-01-27              NaN  Marketing   

            job_title  salary  
0        Data Analyst    6383  
1     Account Manager    7835  
2          Accountant    5900  
3           Recruiter    6089  
4  Content Strategist    5852  


## Exploración de los datos

In [75]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   employee_id       10000 non-null  int64 
 1   gender            10000 non-null  object
 2   birth_date        10000 non-null  object
 3   hire_date         10000 non-null  object
 4   termination_date  3052 non-null   object
 5   department        10000 non-null  object
 6   job_title         10000 non-null  object
 7   salary            10000 non-null  int64 
dtypes: int64(2), object(6)
memory usage: 625.1+ KB


## Conversión de los datos de tipo fecha

In [76]:
cols_fecha = ["birth_date","hire_date", "termination_date"]
df[cols_fecha] = df[cols_fecha].apply(pd.to_datetime)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   employee_id       10000 non-null  int64         
 1   gender            10000 non-null  object        
 2   birth_date        10000 non-null  datetime64[ns]
 3   hire_date         10000 non-null  datetime64[ns]
 4   termination_date  3052 non-null   datetime64[ns]
 5   department        10000 non-null  object        
 6   job_title         10000 non-null  object        
 7   salary            10000 non-null  int64         
dtypes: datetime64[ns](3), int64(2), object(3)
memory usage: 625.1+ KB


## Exploración de KPI's

### Headcount
- **Número de empleados activos:** Se considera a los registros que no cuenten regitrado una fecha de término de vinculo laboral.

In [77]:
personal_activo = df.loc[df["termination_date"].isna()].copy()
print(f"El personal en actividad es de {personal_activo.shape[0]} empleados")

El personal en actividad es de 6948 empleados


## Tasa de rotación anual - Evaluada en el año 2020

### Años de análisis

In [78]:
fecha_inicio = pd.Timestamp("2020-01-01")
fecha_final = pd.Timestamp("2020-12-31")

print(f"Fecha inicio del periodo: {fecha_inicio}")
print(f"Fecha final del periodo: {fecha_final}")

Fecha inicio del periodo: 2020-01-01 00:00:00
Fecha final del periodo: 2020-12-31 00:00:00


### Empleados activos durante el 2020
- Un empleado está activo en 2020 si:
    - Fue contratado antes o durante 2020
    - No terminó o terminó después de 2020

### Índice de rotación de personal - Formulada en la salida de personal
- Evaluada en el año 2020

$$
\text{IRP}_{1} (\%) = \dfrac{\text{Número de salidas en el período}}{\text{Promedio de trabajadores en el período}} \times 100
$$

### Personal en actividad laboral durante el 2020

In [79]:
actividad_2020 = df[
    (df["hire_date"] <= fecha_final) &
    ((df["termination_date"].isna()) | (df["termination_date"] > fecha_final))
]

### Personal de salida durante el 2020

In [80]:
salidas_2020 = df[
    (df["termination_date"] >= fecha_inicio) &
    (df["termination_date"] <= fecha_final)
]

In [81]:
num_salidas = salidas_2020.shape[0]
prom_empleados = (salidas_2020.shape[0] + actividad_2020.shape[0])/2

rotacion_2020 = (num_salidas / prom_empleados)*100 if prom_empleados > 0 else 0
print(f"Rotación anual 2020 es: {rotacion_2020:.2f}%")

Rotación anual 2020 es: 7.76%


### Índice de rotación de personal - formulada en el movimiento global de personal (A + D)

$$
\text{IRP}_{2} (\%) = \dfrac{\text{A +D}}{\text{F1 + F2}} \times 100
$$

- Evaluada en el año 2020
    - A: Número de personas contratadas durante el período.
    - D: Número de personas desvinculadas durante el período.
    - F1: Número de trabajadores al inicio del período.
    - F2: Número de trabajadores al final del período.

In [82]:
# personas contratadas durante el período
A = df[
    (df["hire_date"] >= fecha_inicio) &
    (df["hire_date"] <= fecha_final)
    ].shape[0]
# personas desvinculadas durante el período
D = df[
    (df["termination_date"] >= fecha_inicio) & 
    (df["termination_date"] <= fecha_final)
    ].shape[0]
# trabajadores al inicio del período.
F1 = df[
    (df["hire_date"] <= fecha_inicio) &
    ((df["termination_date"].isna()) | (df["termination_date"] >= fecha_inicio))
].shape[0]
# trabajadores al final del período
F2 = df[
    (df["hire_date"] <= fecha_final) &
    ((df["termination_date"].isna()) | (df["termination_date"] >= fecha_final))
].shape[0]

In [83]:
rotacion_2020 = round(((A+D)/2)*100 / ((F1+F2)/2),2)
print(f"Rotación anual 2020 es: {rotacion_2020:.2f}%")

Rotación anual 2020 es: 7.83%


$$
\text{IRP}_{1} \approx \text{IRP}_{2}
$$
$$
\text{7.76} \approx \text{7.83}
$$

- **Conclusión:** De acuerdo a los resultados se podría inferir que los ingresos y las salidas de personal fueron numericamente similares durante el año 2020, lo que indica que por cada trabajador que se desvinculó, ingresó aproximadamente uno, manteniéndose estable la dotación de personal.

## Edad promedio
### Calculando la fecha actual

In [84]:
fecha_actual = pd.Timestamp.today().normalize()

### Calculando la edad actual correctamente
- (corrigiendo si aún no cumplió años este año)

In [85]:
personal_activo["edad"] = (fecha_actual.year - personal_activo["birth_date"].dt.year
             - (
                 (fecha_actual.month < personal_activo["birth_date"].dt.month)
                 | (
                     (fecha_actual.month == personal_activo["birth_date"].dt.month)
                     & (fecha_actual.day < personal_activo["birth_date"].dt.day)
                 )
             ).astype(int)
             )
edad_promedio = personal_activo["edad"].mean()
print(f"La edad promedio del personal es: {edad_promedio:.2f}") 

La edad promedio del personal es: 41.82


## Distribución de personal activo por género

In [86]:
conteo = personal_activo.groupby("gender").size()
conteo = conteo.rename({"F": "Femenino", "M":"Masculino"}).to_frame().reset_index()
conteo.columns = ["Género","Conteo"]
conteo

,Género,Conteo
0,Femenino,3528
1,Masculino,3420


### Promedio de la edad del personal por género

In [87]:
edad_genero = personal_activo.groupby("gender")["edad"].mean()
edad_genero = edad_genero.rename({"F": "Femenino", "M":"Masculino"}).to_frame().reset_index()
edad_genero.columns = ["Género","Edad_promedio"]
edad_genero

,Género,Edad_promedio
0,Femenino,41.764172
1,Masculino,41.885673
